# Stage 2 follow-up: Health Check
## ISM 6562 — Final Project — MediStream Telehealth
---
Single-pane sanity check for all Stage 2 outputs (base + follow-ups). Run after `02-spark-transforms.ipynb` and the `02b`–`02f` follow-ups to confirm every expected table exists with the right schema and a non-zero row count.

Useful for:
- A quick "did everything run?" check before handing off to Stage 3
- Catching schema drift if someone re-runs an upstream notebook with different config
- Confirming partition layout for the repartitioned curated tables

## Setup

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName('MediStream-Stage2g-HealthCheck')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '512m')
    .config('spark.driver.memory', '512m')
    .getOrCreate())

HDFS_CURATED   = 'hdfs://namenode:9000/medistream/curated'
HDFS_ANALYTICS = 'hdfs://namenode:9000/medistream/analytics'
LOCAL_CURATED   = '/home/jovyan/data/curated'
LOCAL_ANALYTICS = '/home/jovyan/data/analytics'

try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_ANALYTICS}/_probe')
    CURATED, ANALYTICS = HDFS_CURATED, HDFS_ANALYTICS
    print('Using HDFS')
except Exception:
    CURATED, ANALYTICS = LOCAL_CURATED, LOCAL_ANALYTICS
    print('HDFS unavailable, using local mount')

## Expected outputs and their required columns

Defined declaratively so the check is easy to extend if someone adds a new table.

In [ ]:
EXPECTED = {
    # zone -> {table_name: required_columns}
    'curated': {
        'appointments':       {'appointment_id', 'patient_id', 'physician_id', 'specialty', 'status', 'visit_type', 'scheduled_time'},
        'patient_vitals':     {'patient_id', 'measurement_type', 'value', 'unit', 'device_type'},
        'session_quality':    {'session_id', 'appointment_id', 'latency_ms', 'packet_loss_pct', 'audio_quality_score', 'device_type', 'os'},
        'patient_feedback':   {'feedback_id', 'appointment_id', 'patient_id', 'rating', 'nps_score'},
        'physician_schedule': {'physician_id', 'date', 'max_appointments', 'actual_appointments'},
    },
    'analytics': {
        # base notebook outputs
        'no_show_features':    set(),  # schema is dynamic in base notebook — just check existence
        'physician_perf':      {'physician_id', 'total_appointments'},
        'session_quality_agg': {'session_count'},
        'utilization_rates':   {'physician_id', 'utilization_pct'},
        'patient_journey':     {'patient_id', 'total_appointments'},
        # follow-up outputs (02b–02e)
        'no_show_breakdown':                 {'specialty', 'visit_type', 'day_of_week', 'time_of_day', 'no_show_rate_pct'},
        'quality_by_device_os':              {'device_type', 'os', 'session_count', 'quality_pass_rate_pct'},
        'patient_history_scores':            {'patient_id', 'patient_history_score', 'engagement_tier'},
        'physician_quality_adjusted_volume': {'physician_id', 'quality_adjusted_volume', 'avg_rating'},
        'degraded_sessions':                 {'session_id', 'degraded_severity', 'degraded_latency', 'degraded_packet_loss', 'degraded_audio'},
    },
}

## Run the check

In [ ]:
results = []
for zone, tables in EXPECTED.items():
    base = CURATED if zone == 'curated' else ANALYTICS
    for name, required_cols in tables.items():
        path = f'{base}/{name}'
        try:
            df = spark.read.parquet(path)
            cols = set(df.columns)
            missing = required_cols - cols
            row_count = df.count()
            if missing:
                status = f'FAIL — missing cols: {sorted(missing)}'
            elif row_count == 0:
                status = 'FAIL — zero rows'
            else:
                status = 'OK'
            results.append((zone, name, row_count, len(cols), status))
        except Exception as e:
            results.append((zone, name, 0, 0, f'FAIL — read error: {type(e).__name__}'))

print(f'{"zone":<10s} {"table":<40s} {"rows":>10s} {"cols":>5s}  status')
print('-' * 100)
for zone, name, rows, cols, status in results:
    print(f'{zone:<10s} {name:<40s} {rows:>10,} {cols:>5d}  {status}')

n_ok = sum(1 for r in results if r[4] == 'OK')
print(f'\n{n_ok}/{len(results)} tables healthy')
assert n_ok == len(results), f'{len(results) - n_ok} tables failed health check'

## Confirm partition layout for repartitioned curated tables

After `02f-repartition-curated.ipynb` runs, these three tables should have nested partition directories on disk.

In [ ]:
sc = spark.sparkContext
fs = sc._jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())

EXPECTED_PARTITIONS = {
    'curated/appointments':    'specialty=',
    'curated/session_quality': 'device_type=',
    'curated/patient_vitals':  'measurement_type=',
    'analytics/no_show_breakdown':    'specialty=',
    'analytics/quality_by_device_os': 'device_type=',
    'analytics/patient_history_scores': 'engagement_tier=',
    'analytics/degraded_sessions': 'degraded_severity=',
}

for rel_path, expected_prefix in EXPECTED_PARTITIONS.items():
    base = CURATED if rel_path.startswith('curated') else ANALYTICS
    table_name = rel_path.split('/', 1)[1]
    full = f'{base}/{table_name}'
    try:
        path = sc._jvm.org.apache.hadoop.fs.Path(full)
        statuses = fs.listStatus(path)
        names = [s.getPath().getName() for s in statuses if s.isDirectory()]
        partitioned = [n for n in names if n.startswith(expected_prefix)]
        if partitioned:
            print(f'  OK   {rel_path:<40s} {len(partitioned)} partitions starting with {expected_prefix!r}')
        else:
            print(f'  WARN {rel_path:<40s} no partition dirs starting with {expected_prefix!r} (run 02f / 02b–02e?)')
    except Exception as e:
        print(f'  ERR  {rel_path:<40s} {type(e).__name__}: {e}')